# 02 — Comparaison Baseline vs Improved

**EFREI 2025-2026 · Solution Delivery · Filière Data**

> **Position non clinique.** Ce dépôt n'est pas un dispositif médical. Les sorties ne constituent pas un avis médical.

## Objectif

Comparer les deux versions du prompt sur le dataset synthétique :

| Version | Modèle | Spécificités |
|---|---|---|
| **baseline** | claude-haiku (toy en fallback) | Prompt simple, pas de confidence gate |
| **improved** | claude-sonnet (toy en fallback) | Protocole d'analyse, confidence gate 0.60 |

Sorties produites :
- `eval/outputs/before_after_summary.csv` — comparaison agrégée
- `eval/outputs/error_register.csv` — analyse cas par cas

In [ ]:
import sys, csv, json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

with (ROOT / 'data' / 'synthetic_cases.csv').open(newline='', encoding='utf-8') as f:
    cases = list(csv.DictReader(f))

print(f'{len(cases)} cas chargés')

## 1 — Évaluation des deux modes

In [ ]:
from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails
from src.metrics import summarize_metrics

def evaluate_mode(mode: str) -> tuple[list[dict], dict]:
    """Run toy inference on all cases for a given mode, return (rows, metrics)."""
    rows = []
    for case in cases:
        image_path = ROOT / case['image_path']
        pred = apply_safety_guardrails(toy_predict(image_path, mode=mode))
        rows.append({
            'case_id':         case['case_id'],
            'label':           case['label'],
            'predicted_class': pred['predicted_class'],
            'confidence':      pred['confidence'],
            'image_quality':   pred['image_quality'],
            'json_valid':      True,
            'warning':         pred.get('warning', ''),
            'model_name':      pred.get('model_name', ''),
        })
    metrics = summarize_metrics(rows)
    return rows, metrics

rows_base, m_base = evaluate_mode('baseline')
rows_imp,  m_imp  = evaluate_mode('improved')

print('baseline :', {k: round(v, 3) for k, v in m_base.items()})
print('improved :', {k: round(v, 3) for k, v in m_imp.items()})

## 2 — Tableau de comparaison before / after

In [ ]:
METRICS = ['accuracy', 'macro_f1', 'json_valid_rate', 'warning_rate', 'uncertain_rate']

summary = pd.DataFrame([
    {'mode': 'baseline', **{k: m_base[k] for k in METRICS}},
    {'mode': 'improved', **{k: m_imp[k]  for k in METRICS}},
])

# Delta improved vs baseline
for k in ['accuracy', 'macro_f1']:
    summary[f'delta_{k}'] = summary[k].diff()

display(summary.style.format({
    'accuracy': '{:.1%}', 'macro_f1': '{:.1%}',
    'json_valid_rate': '{:.1%}', 'warning_rate': '{:.1%}',
    'uncertain_rate': '{:.1%}',
    'delta_accuracy': '{:+.1%}', 'delta_macro_f1': '{:+.1%}',
}))

## 3 — Analyse des prompts

Affiche les deux fichiers de prompt pour comprendre ce qui change entre baseline et improved.

In [ ]:
prompts_dir = ROOT / 'prompts'

for fname in ['baseline_prompt.txt', 'improved_prompt.txt']:
    p = prompts_dir / fname
    if p.exists():
        print(f'=== {fname} ===')
        print(p.read_text(encoding='utf-8'))
        print()

## 4 — Registre d'erreurs commenté

In [ ]:
ERROR_TAXONOMY = {
    'FN': 'Faux négatif — anomalie présente prédite normale',
    'FP': 'Faux positif — image normale prédite suspecte',
    'UA': 'Incertitude acceptable — signes faibles ou image limitée',
    'JF': 'JSON format error — sortie non exploitable',
    'HT': 'Hallucination textuelle — mention d\'un signe non visible',
    'OK': 'Correct',
}

def classify_error(label: str, predicted: str, quality: str = 'good') -> str:
    if label == predicted:
        return 'OK'
    if label == 'suspected_opacity' and predicted == 'normal':
        return 'FN'
    if label == 'normal' and predicted == 'suspected_opacity':
        return 'FP'
    return 'UA'

by_id_imp = {r['case_id']: r for r in rows_imp}
register = []

for r in rows_base:
    cid   = r['case_id']
    r_imp = by_id_imp.get(cid, {})
    err_b = classify_error(r['label'], r['predicted_class'], r['image_quality'])
    err_i = classify_error(r['label'], r_imp.get('predicted_class', '—'), r_imp.get('image_quality', 'good'))

    if err_b == 'OK' and err_i == 'OK':
        comment = 'Les deux modes classifient correctement.'
    elif err_b == 'OK' and err_i != 'OK':
        comment = f'Régression : baseline OK, improved produit {err_i}.'
    elif err_b != 'OK' and err_i == 'OK':
        comment = f'Amélioration : baseline produit {err_b}, improved OK.'
    elif 'UA' in (err_b, err_i):
        comment = 'Incertitude acceptable — qualité image ou cas ambigu.'
    else:
        comment = f'Erreur persistante : baseline={err_b}, improved={err_i}.'

    register.append({
        'case_id':       cid,
        'ground_truth':  r['label'],
        'baseline_pred': r['predicted_class'],
        'baseline_conf': r['confidence'],
        'baseline_error': err_b,
        'improved_pred': r_imp.get('predicted_class', '—'),
        'improved_conf': r_imp.get('confidence', '—'),
        'improved_error': err_i,
        'comment':       comment,
    })

df_reg = pd.DataFrame(register)
display(df_reg)

print('\nRépartition codes erreur — baseline:')
print(df_reg['baseline_error'].value_counts().to_string())
print('\nRépartition codes erreur — improved:')
print(df_reg['improved_error'].value_counts().to_string())

## 5 — Sauvegarde

In [ ]:
out_dir = ROOT / 'eval' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

# before_after_summary.csv — requis par le test de smoke du prof
summary_rows = [
    {'mode': 'baseline', **{k: m_base[k] for k in METRICS}},
    {'mode': 'improved', **{k: m_imp[k]  for k in METRICS}},
]
pd.DataFrame(summary_rows).to_csv(out_dir / 'before_after_summary.csv', index=False)

# Métriques JSON individuelles
(out_dir / 'baseline_metrics.json').write_text(json.dumps(m_base, indent=2), encoding='utf-8')
(out_dir / 'improved_metrics.json').write_text(json.dumps(m_imp, indent=2), encoding='utf-8')

# Registre d'erreurs
df_reg.to_csv(out_dir / 'error_register.csv', index=False)
(out_dir / 'error_register.json').write_text(
    json.dumps({'taxonomy': ERROR_TAXONOMY, 'cases': register}, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

print('Fichiers sauvegardés dans', out_dir)
for f in sorted(out_dir.glob('*.csv')) + sorted(out_dir.glob('*.json')):
    print(' ', f.name)

## Limites de ce benchmark

| Limite | Impact |
|---|---|
| Dataset synthétique (N=30) | Insuffisant pour conclusions statistiques robustes |
| Labels par construction | Aucun radiologue n'a validé les ground truths |
| Toy predictor | Classe par nom de fichier — non généralisable |
| Reproductibilité VLM | Réponses Claude varient entre appels |
| Absence de validation clinique | Ne jamais utiliser en conditions réelles |

> *Ces limites doivent être explicitement mentionnées dans la soutenance.*